# Section 9 - Advanced Statistical Analysis

This notebook covers RFM Customer Segmentation, K-Means Clustering, and Cohort Analysis.

In [ ]:
# ================================
# 1. Imports and setup
# ================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

In [ ]:
# ================================
# 2. Load the cleaned data
# ================================
df = pd.read_csv('../data/processed/cleaned_data.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df.head()

### RFM Analysis
Recency, Frequency, and Monetary Value segmentation.

In [ ]:
# ================================
# 3. Calculate RFM Metrics
# ================================
snapshot_date = df['InvoiceDate'].max() + timedelta(days=1)

# Aggregate data on CustomerID level
rfm = df.dropna(subset=['CustomerID']).groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'Revenue': 'sum'
}).reset_index()

rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'Revenue': 'Monetary'
}, inplace=True)

rfm.head()

In [ ]:
# ================================
# 4. RFM Quartile Scoring
# ================================
r_labels = range(4, 0, -1) # Recency: Lower is better (4 is best)
f_labels = range(1, 5)     # Frequency: Higher is better
m_labels = range(1, 5)     # Monetary: Higher is better

r_quartiles = pd.qcut(rfm['Recency'], q=4, labels=r_labels)
f_quartiles = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=f_labels)
m_quartiles = pd.qcut(rfm['Monetary'], q=4, labels=m_labels)

rfm = rfm.assign(R=r_quartiles, F=f_quartiles, M=m_quartiles)
rfm['RFM_Score'] = rfm[['R', 'F', 'M']].astype(str).agg(''.join, axis=1)

# Create segments based on R and F scores
def join_rfm(x): return str(x['R']) + str(x['F'])
rfm['RF_Segment'] = rfm.apply(join_rfm, axis=1)

rfm.head()

### K-Means Clustering
Using machine learning to find distinct customer groups based on RFM features.

In [ ]:
# ================================
# 5. Data Preprocessing for K-Means
# ================================
# Handle right-skewed data with log transformation
# Adding 1 to avoid log(0)
rfm_log = rfm[['Recency', 'Frequency', 'Monetary']].apply(lambda x: np.log(x + 1))

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)
rfm_scaled = pd.DataFrame(rfm_scaled, index=rfm.index, columns=rfm_log.columns)

In [ ]:
# ================================
# 6. Determining Optimal K (Elbow Method)
# ================================
inertia = []
k_range = range(1, 11)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(rfm_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia, marker='o')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.xticks(k_range)
plt.show()

In [ ]:
# ================================
# 7. Fit K-Means and Profile Clusters
# ================================
# From the elbow plot, K=3 or 4 is a good choice. We use 4.
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

cluster_summary = rfm.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': ['mean', 'count']
}).round(1)

display(cluster_summary)

### Cohort Analysis
Tracking customer retention across acquisition months.

In [ ]:
# ================================
# 8. Prepare Cohort Data
# ================================
def get_month(x): return x.replace(day=1, hour=0, minute=0, second=0)
df['InvoiceMonth'] = df['InvoiceDate'].apply(get_month)

grouping = df.dropna(subset=['CustomerID']).groupby('CustomerID')['InvoiceMonth']
df['CohortMonth'] = grouping.transform('min')
df = df.dropna(subset=['CohortMonth']) # Drop rows without customer IDs

def get_date_int(df, column):
    year = df[column].dt.year
    month = df[column].dt.month
    return year, month

invoice_year, invoice_month = get_date_int(df, 'InvoiceMonth')
cohort_year, cohort_month = get_date_int(df, 'CohortMonth')

years_diff = invoice_year - cohort_year
months_diff = invoice_month - cohort_month
df['CohortIndex'] = years_diff * 12 + months_diff + 1

In [ ]:
# ================================
# 9. Calculate Retention Matrix
# ================================
cohort_data = df.groupby(['CohortMonth', 'CohortIndex'])['CustomerID'].nunique().reset_index()
cohort_counts = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='CustomerID')

cohort_sizes = cohort_counts.iloc[:,0]
retention = cohort_counts.divide(cohort_sizes, axis=0)

# Format the CohortMonth for better display
retention.index = retention.index.strftime('%Y-%m')

plt.figure(figsize=(15, 8))
sns.heatmap(data=retention, annot=True, fmt='.0%', cmap='Blues', vmin=0.0, vmax=0.5)
plt.title('Customer Retention Matrix by Monthly Cohorts')
plt.ylabel('Cohort Month')
plt.xlabel('Cohort Index (Months since first purchase)')
plt.show()